In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
customers_df = spark.table("project.silver.silver_customers").drop("ingestion_timestamp")
orders_df = spark.table("project.silver.silver_orders").drop("ingestion_timestamp")
order_items_df = spark.table("project.silver.silver_items").drop("ingestion_timestamp")
products_df = spark.table("project.silver.silver_product")
payments_df = spark.table("project.silver.silver_payments").drop("ingestion_timestamp")

In [0]:
delivered_orders = orders_df.filter(F.col("order_status") == "delivered") \
    .dropna(subset=["order_delivered_customer_date"])

In [0]:
display(delivered_orders.groupBy("order_status").count())

order_status,count
delivered,96470


In [0]:
payments_agg = payments_df.groupBy("order_id").agg(
    F.max("is_boleto").alias("is_boleto"),
    F.max("payment_installments_count").alias("installments"),
    F.max("multi_payment_flag").alias("is_multi_payment")
)

display(payments_agg.head(3))

order_id,is_boleto,installments,is_multi_payment
cf30fe76d1505192acee1c6dccb15545,0,2,0
85eef2d342b0de363c45c1bc324729c5,0,3,0
ad4098a257676ea4d394fb3bbbf36ca3,0,2,0


##  Join Orders + Customers (Geography)

In [0]:
master_df = delivered_orders.join(
    customers_df.select("customer_id", "customer_state"),
    on="customer_id",
    how="inner"
)

In [0]:
display(master_df.head(3))

customer_id,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,estimated_delivery_buffer_days,purchase_day_of_week,approval_delay_days,is_weekend_purchase,estimated_delivery_month,expected_delivery_duration_days,customer_state
06b8999e2fba1a1fbc88172c00ba8bc7,00e7ee1b050b8499577073aeb2a297a1,delivered,2017-05-16T15:05:35.000Z,2017-05-16T15:22:12.000Z,2017-05-23T10:47:57.000Z,2017-05-25T10:35:35.000Z,2017-06-05T00:00:00.000Z,20,3,0,0,5,20,SP
154c4ded6991bdfa3cd249d11abf4130,830d8b3e6875ef6165ffc33219ab4fea,delivered,2017-08-13T10:03:36.000Z,2017-08-13T10:24:04.000Z,2017-08-14T18:43:45.000Z,2017-08-19T12:37:49.000Z,2017-09-20T00:00:00.000Z,38,1,0,1,8,38,SC
237098a64674ae89babdc426746260fc,eac4ffbe456464bdc3ce5f001b6439c5,delivered,2017-04-14T11:24:56.000Z,2017-04-14T11:35:23.000Z,2017-04-17T10:43:50.000Z,2017-04-20T17:04:51.000Z,2017-05-10T00:00:00.000Z,26,6,0,0,4,26,PR


In [0]:
master_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- estimated_delivery_buffer_days: integer (nullable = true)
 |-- purchase_day_of_week: integer (nullable = true)
 |-- approval_delay_days: integer (nullable = true)
 |-- is_weekend_purchase: integer (nullable = true)
 |-- estimated_delivery_month: integer (nullable = true)
 |-- expected_delivery_duration_days: integer (nullable = true)
 |-- customer_state: string (nullable = true)



## Join with Order Items (Logistics & Seller context)

In [0]:
master_df = master_df.join(
    order_items_df.select(
        "order_id", "product_id", "seller_id", "price", "freight_value",
        "seller_shipment_buffer_days", "freight_ratio", 
        "total_order_items", "seller_popularity_30d"
    ),
    on="order_id",
    how="inner"
)

In [0]:
master_df.limit(3).display()

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,estimated_delivery_buffer_days,purchase_day_of_week,approval_delay_days,is_weekend_purchase,estimated_delivery_month,expected_delivery_duration_days,customer_state,product_id,seller_id,price,freight_value,seller_shipment_buffer_days,freight_ratio,total_order_items,seller_popularity_30d
0005f50442cb953dcd1d21e1fb923495,351d3cb2cee3c7fd0af6616c82df21d3,delivered,2018-07-02T13:59:39.000Z,2018-07-02T14:10:56.000Z,2018-07-03T14:25:00.000Z,2018-07-04T17:28:31.000Z,2018-07-23T00:00:00.000Z,21,2,0,0,7,21,SP,4535b0e1091c278dfd193e5a1d63b39f,ba143b05f0110f0dc71ad71b4466ce92,53.99,11.4,4,0.21115021300240785,1,null
00063b381e2406b52ad429470734ebd5,6a899e55865de6549a58d2c6845e5604,delivered,2018-07-27T17:21:27.000Z,2018-07-27T18:00:06.000Z,2018-07-30T14:52:00.000Z,2018-08-07T13:56:52.000Z,2018-08-07T00:00:00.000Z,11,6,0,0,7,11,SP,f177554ea93259a5b282f24e33f65ab6,8602a61d680a10a82cceeeda0d99ea3d,45.0,12.98,4,0.28844444444444445,1,null
000e63d38ae8c00bbcb5a30573b99628,98884e672c5ba85f4394f2044e1a3eab,delivered,2018-03-23T19:48:26.000Z,2018-03-23T20:07:49.000Z,2018-03-26T21:38:48.000Z,2018-03-27T14:51:47.000Z,2018-04-05T00:00:00.000Z,13,6,0,0,3,13,SP,553e0e7590d3116a072507a3635d2877,1c129092bf23f28a5930387c980c0dfc,47.9,8.88,6,0.18538622129436327,1,null


In [0]:
master_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- estimated_delivery_buffer_days: integer (nullable = true)
 |-- purchase_day_of_week: integer (nullable = true)
 |-- approval_delay_days: integer (nullable = true)
 |-- is_weekend_purchase: integer (nullable = true)
 |-- estimated_delivery_month: integer (nullable = true)
 |-- expected_delivery_duration_days: integer (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- seller_sh

## Join with Products (Physical Complexity)

In [0]:
# Join with Products (Physical Complexity)
master_df = master_df.join(
    products_df.select(
        "product_id", "product_weight_g", "product_volume_cm3",
        "product_density", "is_bulky_item", "product_category_name_english"
    ),
    on="product_id",
    how="inner"
)

## Join with aggregated Payments

In [0]:
master_df = master_df.join(payments_agg, on="order_id", how="left")

In [0]:
master_df.limit(3).display()

order_id,product_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,estimated_delivery_buffer_days,purchase_day_of_week,approval_delay_days,is_weekend_purchase,estimated_delivery_month,expected_delivery_duration_days,customer_state,seller_id,price,freight_value,seller_shipment_buffer_days,freight_ratio,total_order_items,seller_popularity_30d,product_weight_g,product_volume_cm3,product_density,is_bulky_item,product_category_name_english,is_boleto,installments,is_multi_payment
0005f50442cb953dcd1d21e1fb923495,4535b0e1091c278dfd193e5a1d63b39f,351d3cb2cee3c7fd0af6616c82df21d3,delivered,2018-07-02T13:59:39.000Z,2018-07-02T14:10:56.000Z,2018-07-03T14:25:00.000Z,2018-07-04T17:28:31.000Z,2018-07-23T00:00:00.000Z,21,2,0,0,7,21,SP,ba143b05f0110f0dc71ad71b4466ce92,53.99,11.4,4,0.21115021300240785,1,null,850,1827,0.4652435686918446,0,books_technical,0,1,0
00063b381e2406b52ad429470734ebd5,f177554ea93259a5b282f24e33f65ab6,6a899e55865de6549a58d2c6845e5604,delivered,2018-07-27T17:21:27.000Z,2018-07-27T18:00:06.000Z,2018-07-30T14:52:00.000Z,2018-08-07T13:56:52.000Z,2018-08-07T00:00:00.000Z,11,6,0,0,7,11,SP,8602a61d680a10a82cceeeda0d99ea3d,45.0,12.98,4,0.28844444444444445,1,null,200,2816,0.07102272727272728,0,fashion_bags_accessories,0,5,0
000e63d38ae8c00bbcb5a30573b99628,553e0e7590d3116a072507a3635d2877,98884e672c5ba85f4394f2044e1a3eab,delivered,2018-03-23T19:48:26.000Z,2018-03-23T20:07:49.000Z,2018-03-26T21:38:48.000Z,2018-03-27T14:51:47.000Z,2018-04-05T00:00:00.000Z,13,6,0,0,3,13,SP,1c129092bf23f28a5930387c980c0dfc,47.9,8.88,6,0.18538622129436327,1,null,800,8000,0.1,0,bed_bath_table,0,1,0


## TARGET LABELING & FINAL FEATURE SELECTION

### Target Variable: 1 if actual delivery date is after estimated date

---

In [0]:
gold_df = master_df.withColumn(
    "is_late", 
    F.when(F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date"), 1).otherwise(0)
)

In [0]:
display(gold_df.limit(3))

order_id,product_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,estimated_delivery_buffer_days,purchase_day_of_week,approval_delay_days,is_weekend_purchase,estimated_delivery_month,expected_delivery_duration_days,customer_state,seller_id,price,freight_value,seller_shipment_buffer_days,freight_ratio,total_order_items,seller_popularity_30d,product_weight_g,product_volume_cm3,product_density,is_bulky_item,product_category_name_english,is_boleto,installments,is_multi_payment,is_late
0005f50442cb953dcd1d21e1fb923495,4535b0e1091c278dfd193e5a1d63b39f,351d3cb2cee3c7fd0af6616c82df21d3,delivered,2018-07-02T13:59:39.000Z,2018-07-02T14:10:56.000Z,2018-07-03T14:25:00.000Z,2018-07-04T17:28:31.000Z,2018-07-23T00:00:00.000Z,21,2,0,0,7,21,SP,ba143b05f0110f0dc71ad71b4466ce92,53.99,11.4,4,0.21115021300240785,1,null,850,1827,0.4652435686918446,0,books_technical,0,1,0,0
00063b381e2406b52ad429470734ebd5,f177554ea93259a5b282f24e33f65ab6,6a899e55865de6549a58d2c6845e5604,delivered,2018-07-27T17:21:27.000Z,2018-07-27T18:00:06.000Z,2018-07-30T14:52:00.000Z,2018-08-07T13:56:52.000Z,2018-08-07T00:00:00.000Z,11,6,0,0,7,11,SP,8602a61d680a10a82cceeeda0d99ea3d,45.0,12.98,4,0.28844444444444445,1,null,200,2816,0.07102272727272728,0,fashion_bags_accessories,0,5,0,1
000e63d38ae8c00bbcb5a30573b99628,553e0e7590d3116a072507a3635d2877,98884e672c5ba85f4394f2044e1a3eab,delivered,2018-03-23T19:48:26.000Z,2018-03-23T20:07:49.000Z,2018-03-26T21:38:48.000Z,2018-03-27T14:51:47.000Z,2018-04-05T00:00:00.000Z,13,6,0,0,3,13,SP,1c129092bf23f28a5930387c980c0dfc,47.9,8.88,6,0.18538622129436327,1,null,800,8000,0.1,0,bed_bath_table,0,1,0,0


In [0]:
gold_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- estimated_delivery_buffer_days: integer (nullable = true)
 |-- purchase_day_of_week: integer (nullable = true)
 |-- approval_delay_days: integer (nullable = true)
 |-- is_weekend_purchase: integer (nullable = true)
 |-- estimated_delivery_month: integer (nullable = true)
 |-- expected_delivery_duration_days: integer (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- seller_sh

In [0]:
final_columns = [
    "order_id",
    "is_late",                         # TARGET
    "customer_state",                  # Geographic Risk
    "purchase_day_of_week",            # Temporal (Weekend lag)
    "is_weekend_purchase",             # Temporal
    "estimated_delivery_month",        # Seasonality (Black Friday/Holiday risk)
    "approval_delay_days",             # Payment clearing friction
    "estimated_delivery_buffer_days",  # Platform optimism (Tight deadlines = High risk)
    "expected_delivery_duration_days", # Total delivery window promised
    "seller_shipment_buffer_days",     # Seller handling time risk
    "freight_ratio",                   # Shipping speed vs cost proxy
    "total_order_items",               # Consolidation complexity
    "seller_popularity_30d",           # Seller bottleneck risk
    "product_weight_g",                # Carrier tier risk
    "product_volume_cm3",              # Logistics volume constraint
    "product_density",                 # Weight/Volume handling ratio
    "is_bulky_item",                   # Specialized freight requirement
    "product_category_name_english",   # Historical category delay risk
    "is_boleto",                       # High-risk payment method (late start)
    "installments",                    # Ticket value/Anti-fraud proxy
    "is_multi_payment"                 # Payment processing complexity
]

In [0]:
delivery_delay_ml_data = gold_df.select(*final_columns).fillna({
    "installments": 1,
    "is_boleto": 0,
    "is_multi_payment": 0,
    "product_category_name_english": "unknown",
    "approval_delay_days": 0,
    "seller_popularity_30d": 0
})

print(f"Final Dataset Row Count: {delivery_delay_ml_data.count()}")


Final Dataset Row Count: 106225


In [0]:
display(delivery_delay_ml_data.limit(5))

order_id,is_late,customer_state,purchase_day_of_week,is_weekend_purchase,estimated_delivery_month,approval_delay_days,estimated_delivery_buffer_days,expected_delivery_duration_days,seller_shipment_buffer_days,freight_ratio,total_order_items,seller_popularity_30d,product_weight_g,product_volume_cm3,product_density,is_bulky_item,product_category_name_english,is_boleto,installments,is_multi_payment
0005f50442cb953dcd1d21e1fb923495,0,SP,2,0,7,0,21,21,4,0.21115021300240785,1,0,850,1827,0.4652435686918446,0,books_technical,0,1,0
00063b381e2406b52ad429470734ebd5,1,SP,6,0,7,0,11,11,4,0.28844444444444445,1,0,200,2816,0.07102272727272728,0,fashion_bags_accessories,0,5,0
000e63d38ae8c00bbcb5a30573b99628,0,SP,6,0,3,0,13,13,6,0.18538622129436327,1,0,800,8000,0.1,0,bed_bath_table,0,1,0
003a7f59d7e08a9c61d9e2881fe6459c,0,DF,1,1,8,1,10,10,3,0.1284033613445378,1,0,700,13832,0.05060728744939271,0,housewares,0,10,0
005cad6157eadc7f1f09917607f1704a,0,SP,7,1,4,1,25,25,5,0.15458333333333332,1,0,3300,59280,0.05566801619433198,1,sports_leisure,0,2,0


##  Save the Gold table

In [0]:
print("Saving Gold table to project.gold.delivery_delay...")
delivery_delay_ml_data.write.format("delta").mode("overwrite") \
    .option("mergeSchema", "false") \
    .saveAsTable("project.gold.delivery_delay_enhanced")

print("Pipeline complete. Real-time dataset created.")

Saving Gold table to project.gold.delivery_delay...
Pipeline complete. Real-time dataset created.
